# InstantSfM Google Colab Notebook

This notebook allows you to run [InstantSfM](https://github.com/cre185/InstantSfM) on your datasets directly from Google Drive.

## 1. Setup Environment
Run the cell below to connect to your Google Drive and install all required dependencies (this may take a few minutes).

In [ ]:
# @title Mount Google Drive & Install Dependencies

from google.colab import drive
drive.mount('/content/drive')

import os
import subprocess
import sys

def run_cmd(cmd):
    print(f"Running: {cmd}")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Error: {result.stderr}")
    else:
        print(result.stdout)
        
print("Installing system dependencies (COLMAP, etc.)...")
!sudo apt-get update && apt-get install -y colmap libsuitesparse-dev

print("Installing Python dependencies...")
!pip install ninja scikit-sparse

print("Installing cuDSS...")
# Install cuDSS (CUDA 12 version for PyTorch 2.x on Colab)
!wget -q https://developer.download.nvidia.com/compute/cudss/redist/libcudss/linux-x86_64/libcudss-linux-x86_64-0.1.0.0_cuda12-archive.tar.xz -O /tmp/cudss.tar.xz
!tar -xJf /tmp/cudss.tar.xz -C /opt
!cp -r /opt/libcudss-linux-x86_64-0.1.0.0_cuda12-archive/include/* /usr/local/cuda/include/
!cp -r /opt/libcudss-linux-x86_64-0.1.0.0_cuda12-archive/lib/* /usr/local/cuda/lib64/

print("Cloning InstantSfM repository...")
if not os.path.exists('/content/InstantSfM'):
    !git clone https://github.com/cre185/instantsfm.git --recursive /content/InstantSfM

os.chdir('/content/InstantSfM')

print("Installing InstantSfM...")
!pip install -e .

print("Installing BAE...")
!pip install git+https://github.com/zitongzhan/bae.git

print("\n✅ Environment setup complete!")


## 2. Run InstantSfM Pipeline
Specify the path to your dataset in Google Drive. The dataset should have a subfolder named `images/` containing your images.

In [ ]:
# @title Execute InstantSfM

Dataset_Path = '/content/drive/MyDrive/MyDataset' # @param {type:"string"}
Manual_Config_Name = '' # @param {type:"string"}
Run_Feature_Extraction = True # @param {type:"boolean"}
Run_SfM = True # @param {type:"boolean"}
Run_3DGS = False # @param {type:"boolean"}
Export_TXT = True # @param {type:"boolean"}

import os

if not os.path.exists(Dataset_Path):
    print(f"❌ Error: Dataset path '{Dataset_Path}' does not exist. Please check the path.")
else:
    print(f"✅ Dataset found at '{Dataset_Path}'")
    
    config_arg = f"--manual_config_name {Manual_Config_Name}" if Manual_Config_Name else ""
    export_arg = "--export_txt" if Export_TXT else ""
    
    if Run_Feature_Extraction:
        print("\n--- Running Feature Extraction ---")
        cmd = f"ins-feat --data_path '{Dataset_Path}' {config_arg}"
        !{cmd}
        
    if Run_SfM:
        print("\n--- Running Structure from Motion (SfM) ---")
        cmd = f"ins-sfm --data_path '{Dataset_Path}' {config_arg} {export_arg}"
        !{cmd}
        
    if Run_3DGS:
        print("\n--- Running 3D Gaussian Splatting (3DGS) ---")
        cmd = f"ins-gs --data_path '{Dataset_Path}' {config_arg}"
        !{cmd}
        
    print("\n🎉 Pipeline execution completed!")


## 3. Results
Your results are saved in the dataset directory you provided. For example, SfM results will be under `sparse/0/` in your dataset folder.

In [ ]:
# @title Zip and Download Results

Zip_Sparse_Results = True # @param {type:"boolean"}
Zip_3DGS_Results = False # @param {type:"boolean"}
Download_Directly = True # @param {type:"boolean"}

import os
import shutil
from google.colab import files

if Zip_Sparse_Results:
    sparse_dir = os.path.join(Dataset_Path, 'sparse', '0')
    if os.path.exists(sparse_dir):
        zip_path = os.path.join(Dataset_Path, 'sparse_results.zip')
        print(f"Zipping sparse results to {zip_path}...")
        !zip -r "{zip_path}" "{sparse_dir}"
        if Download_Directly:
            files.download(zip_path)
    else:
        print(f"Sparse results not found at {sparse_dir}")

if Zip_3DGS_Results:
    output_dir = os.path.join(Dataset_Path, 'output')
    if os.path.exists(output_dir):
        zip_path = os.path.join(Dataset_Path, '3dgs_results.zip')
        print(f"Zipping 3DGS results to {zip_path}...")
        !zip -r "{zip_path}" "{output_dir}"
        if Download_Directly:
            files.download(zip_path)
    else:
        print(f"3DGS results not found at {output_dir}")
